# Session 5 — Data Engineering Patterns
**Phase 3+4 Combined | Project Prometheus**

---

## 🎯 Why This Matters in Quant Finance

A research model is only as good as the data it runs on. Production quant systems fail in boring, unglamorous ways:

- API rate limits hit at 3am, pipeline crashes, no data for Monday's model
- A malformed response silently writes NaN to 20 rows — backtest looks fine, live trade is garbage
- Pipeline runs twice with slightly different parameters, outputs differ by 0.003% — which run is right?

This session is about building infrastructure that **doesn't fail silently**.

> 💡 The Phase 2 Capstone described a full production pipeline. This session gives you the engineering patterns that make each component resilient — so even if you don't build the full Capstone, you know how to apply these patterns when you need them.


---
## 1️⃣ API Ingestion — Retry Logic & Rate Limiting

Financial data APIs fail. Networks timeout. Rate limits hit. The response is sometimes malformed.

**Exponential backoff:** Wait 1s → 2s → 4s → 8s between retries. Avoids hammering a struggling server.

```
Attempt 1: fail → wait 1s
Attempt 2: fail → wait 2s
Attempt 3: fail → wait 4s
Attempt 4: success ✅
```


In [ ]:
import time
import logging
import functools
import yfinance as yf
import pandas as pd
import numpy as np
from typing import Optional

# Set up logging — always log to file in production, not just console
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

def retry_with_backoff(max_retries=3, base_delay=1.0, exceptions=(Exception,)):
    # Decorator: retry a function with exponential backoff on failure
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_retries + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == max_retries:
                        logger.error(f"{func.__name__} failed after {max_retries} retries: {e}")
                        raise
                    wait = base_delay * (2 ** attempt)
                    logger.warning(f"{func.__name__} attempt {attempt+1} failed: {e}. Retrying in {wait}s")
                    time.sleep(wait)
        return wrapper
    return decorator

@retry_with_backoff(max_retries=3, base_delay=0.5)
def fetch_prices(ticker: str, start: str, end: Optional[str] = None) -> pd.Series:
    # Fetch closing prices for a single ticker with retry logic
    data = yf.download(ticker, start=start, end=end,
                       auto_adjust=True, progress=False)['Close']
    if data.empty:
        raise ValueError(f"No data returned for {ticker}")
    return data.squeeze()

# Test it
try:
    spy = fetch_prices('SPY', start='2023-01-01')
    logger.info(f"SPY: {len(spy)} rows, {spy.index[0].date()} to {spy.index[-1].date()}")
    print(f"Fetched {len(spy)} rows for SPY")
except Exception as e:
    print(f"Failed: {e}")


In [ ]:
# Batch fetcher with rate limiting
import time

def fetch_universe(tickers: list, start: str, delay_between: float = 0.5) -> pd.DataFrame:
    # Fetch multiple tickers with delay between requests to avoid rate limits
    results = {}
    failed = []

    for i, ticker in enumerate(tickers):
        try:
            data = fetch_prices(ticker, start=start)
            results[ticker] = data
            logger.info(f"[{i+1}/{len(tickers)}] {ticker}: OK ({len(data)} rows)")
        except Exception as e:
            logger.warning(f"[{i+1}/{len(tickers)}] {ticker}: FAILED — {e}")
            failed.append(ticker)

        # Rate limiting: don't hammer the API
        if i < len(tickers) - 1:
            time.sleep(delay_between)

    if failed:
        logger.warning(f"Failed tickers: {failed}")

    if not results:
        raise RuntimeError("All tickers failed — check network and API")

    df = pd.concat(results, axis=1)
    logger.info(f"Fetch complete: {len(results)} succeeded, {len(failed)} failed")
    return df

# Fetch a small universe
universe = ['SPY', 'QQQ', 'IWM', 'GLD', 'TLT']
prices = fetch_universe(universe, start='2022-01-01', delay_between=0.2)
print(f"Universe shape: {prices.shape}")
print(prices.tail(3))


---
## 2️⃣ Data Validation — Catch Bugs Before They Corrupt Downstream

Validation runs AFTER ingestion, BEFORE cleaning. Its job is to flag problems, not fix them.

**Golden rule:** Be loud. Raise exceptions or log clearly. Silent failures are far worse than noisy ones.

| Check | What it catches |
|-------|----------------|
| Schema validation | Wrong columns, wrong dtypes, missing expected fields |
| Date coverage | Gaps > threshold, missing recent data |
| Price sanity | Prices ≤ 0, returns > 50% in one day (likely split) |
| Stale data | Last date is N days behind today |
| Consistency | All tickers cover same date range |


In [ ]:
class DataValidator:
    # Validates raw financial data before cleaning

    def __init__(self, max_gap_days=5, max_daily_move=0.3, max_missing_pct=0.05):
        self.max_gap_days    = max_gap_days
        self.max_daily_move  = max_daily_move
        self.max_missing_pct = max_missing_pct
        self.issues          = []

    def _add_issue(self, severity, message):
        self.issues.append({'severity': severity, 'message': message})
        if severity == 'ERROR':
            logger.error(f"VALIDATION ERROR: {message}")
        else:
            logger.warning(f"VALIDATION WARNING: {message}")

    def check_missing(self, df: pd.DataFrame):
        for col in df.columns:
            pct = df[col].isna().mean()
            if pct > self.max_missing_pct:
                self._add_issue('ERROR', f"{col}: {pct*100:.1f}% missing (threshold: {self.max_missing_pct*100:.0f}%)")
            elif pct > 0:
                self._add_issue('WARNING', f"{col}: {pct*100:.2f}% missing rows")

    def check_price_sanity(self, df: pd.DataFrame):
        for col in df.columns:
            series = df[col].dropna()
            if (series <= 0).any():
                n_neg = (series <= 0).sum()
                self._add_issue('ERROR', f"{col}: {n_neg} non-positive prices")
            rets = series.pct_change().abs()
            large_moves = rets[rets > self.max_daily_move]
            if len(large_moves) > 0:
                self._add_issue('WARNING', f"{col}: {len(large_moves)} daily moves > {self.max_daily_move*100:.0f}% — possible unadjusted split")

    def check_staleness(self, df: pd.DataFrame, max_age_days=5):
        latest = df.index[-1]
        age = (pd.Timestamp.today() - latest).days
        if age > max_age_days:
            self._add_issue('WARNING', f"Data is {age} days old (last: {latest.date()})")

    def check_date_coverage(self, df: pd.DataFrame):
        business_days = pd.bdate_range(df.index.min(), df.index.max())
        coverage = len(df.index) / len(business_days)
        if coverage < 0.95:
            self._add_issue('WARNING', f"Coverage {coverage*100:.1f}% of expected business days")

    def validate(self, df: pd.DataFrame) -> bool:
        self.issues = []
        self.check_missing(df)
        self.check_price_sanity(df)
        self.check_staleness(df)
        self.check_date_coverage(df)

        errors   = [i for i in self.issues if i['severity'] == 'ERROR']
        warnings = [i for i in self.issues if i['severity'] == 'WARNING']

        print(f"Validation: {len(errors)} errors, {len(warnings)} warnings")
        if errors:
            print("\nERRORS (must fix):")
            for e in errors: print(f"  ❌ {e['message']}")
        if warnings:
            print("\nWARNINGS (review):")
            for w in warnings: print(f"  ⚠️  {w['message']}")

        return len(errors) == 0  # Pass if no errors (warnings OK)

validator = DataValidator()
passed = validator.validate(prices)
print(f"\nValidation {'PASSED ✅' if passed else 'FAILED ❌'}")


---
## 3️⃣ Logging — What Good Logging Looks Like

The difference between debugging in 5 minutes and debugging for 3 hours is logging.

**Levels:**
| Level | Use for |
|-------|---------|
| `DEBUG` | Detailed diagnostic info (verbose — off in production) |
| `INFO` | Normal operation milestones: "Fetched 252 rows for SPY" |
| `WARNING` | Something unexpected but recoverable: "3 missing rows, forward-filled" |
| `ERROR` | Something failed but pipeline can continue |
| `CRITICAL` | Pipeline cannot continue |

**What to log:**
- Start and end of each stage with row counts
- Every data cleaning decision ("forward-filled 3 rows in SPY")
- Every validation failure
- Execution time for slow stages


In [ ]:
import time as time_module

def timed_stage(stage_name):
    # Context manager for timing and logging pipeline stages
    class Timer:
        def __enter__(self):
            self.start = time_module.time()
            logger.info(f"Starting: {stage_name}")
            return self
        def __exit__(self, *args):
            elapsed = time_module.time() - self.start
            logger.info(f"Completed: {stage_name} ({elapsed:.2f}s)")
    return Timer()

# Demo: logged pipeline stages
with timed_stage("Data Fetch"):
    spy = fetch_prices('SPY', start='2022-01-01')

with timed_stage("Validation"):
    validator = DataValidator()
    passed = validator.validate(spy.to_frame())

with timed_stage("Feature Engineering"):
    log_ret = np.log(spy / spy.shift(1))
    roll_vol = log_ret.rolling(20).std() * np.sqrt(252)

print("\nAll stages completed. Check logger output above for timing.")


---
## ✅ Self-Check Questions

1. What is exponential backoff and why is it used for API calls?
   > *Doubling the wait time between retries: 1s → 2s → 4s → 8s. Prevents hammering a struggling server and avoids permanent rate limit bans.*

2. What's the difference between a validation ERROR and a WARNING?
   > *ERROR: pipeline should stop, data is unfit for use (negative prices, >5% missing). WARNING: something unusual but recoverable — log it and continue.*

3. Why should you validate BEFORE cleaning?
   > *Validation on raw data tells you what was wrong with the source. Validation after cleaning only tells you if your cleaning worked. You want both.*

4. What's the purpose of logging row counts at each pipeline stage?
   > *Row counts are a cheap sanity check. If you start with 1000 rows and end with 500, something unexpected happened — the log tells you exactly where.*

---
## 🧪 Optional Tasks

- Add a `max_retries` parameter to `fetch_universe()` and test what happens when you fetch an invalid ticker (e.g., 'XXXINVALID').
- Extend `DataValidator` with a new check: flag tickers where the last date differs by more than 2 business days from the latest date in the universe.
- Add timing logs to the Phase 2 Unit 4 rolling volatility calculation. How long does it take for 10 tickers vs 100?
- Deliberately introduce a bad row (negative price) into a DataFrame. Confirm `DataValidator` catches it.
